In [4]:
import os
os.chdir('/home/dominik/Downloads')

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import scipy.spatial as spatial

class ElasticBubbleCell:
    def __init__(self, position, radius=0.02):
        self.position = np.array(position, dtype=float)
        self.original_position = np.array(position, dtype=float)  # Reference position
        self.velocity = np.zeros(2, dtype=float)
        self.radius = radius
        self.is_t_cell = False
        
        # More nuanced interaction parameters
        self.t_cell_influence_radius = 0.2
        self.influence_strength = 0.4
        self.position_restoration_factor = 0.05
        self.randomness_factor = 0.001

    def interact(self, all_cells):
        # Find nearby T cells
        nearby_t_cells = [
            cell for cell in all_cells 
            if hasattr(cell, 'is_t_cell') and cell.is_t_cell and 
            np.linalg.norm(cell.nucleus - self.position) < self.t_cell_influence_radius
        ]
        
        # Base velocity computation
        if nearby_t_cells:
            # Weighted influence from T cell movement
            influences = []
            for cell in nearby_t_cells:
                # Distance-based weighting
                dist = np.linalg.norm(cell.nucleus - self.position)
                weight = 1 - (dist / self.t_cell_influence_radius)
                
                # Compute weighted velocity influence
                influence = cell.nucleus_velocity * weight * self.influence_strength
                influences.append(influence)
            
            # Average influences
            if influences:
                self.velocity = np.mean(influences, axis=0)
        else:
            # Minimal random movement and position restoration
            self.velocity = (
                np.random.normal(0, self.randomness_factor, 2) +
                (self.original_position - self.position) * self.position_restoration_factor
            )
        
        # Update position
        self.position += self.velocity
        
        # Soft boundary conditions
        self.position = np.clip(self.position, 0, 1)

class TCell:
    def __init__(self, position):
        # Multi-point representation
        self.nucleus = np.array(position, dtype=float)
        self.leading_edge = self.nucleus + np.random.uniform(-0.03, 0.03, 2)
        self.trailing_edge = self.nucleus - np.random.uniform(-0.03, 0.03, 2)
        
        # Movement parameters
        self.persistence = 0.95
        self.probing_intensity = 0.04
        self.exploration_factor = 0.02
        
        # Add velocity initialization
        self.leading_velocity = np.zeros(2)  # Explicitly initialize leading_velocity
        self.nucleus_velocity = np.zeros(2)
        self.trailing_velocity = np.zeros(2)
    
    def probe_environment(self, bubble_cells):
        """Simulate constrained probing in dense environment"""
        probing_force = np.zeros(2)
        deformation_accumulator = 0
        nearby_threshold = 0.02  # Tight interaction radius
        
        for cell in bubble_cells:
            dist = np.linalg.norm(self.leading_edge - cell.position)
            
            if dist < nearby_threshold:
                # Directional interaction
                direction = (self.leading_edge - cell.position) / (dist + 1e-5)
                
                # Progressive force based on proximity
                force_magnitude = (1 - dist / nearby_threshold) * self.probing_intensity
                probing_force += direction * force_magnitude
                
                # Deformation tracking
                deformation_accumulator += force_magnitude
        
        return probing_force, deformation_accumulator
    
    def move(self, bubble_cells):
        # Probing and force computation
        probing_force, deformation = self.probe_environment(bubble_cells)
        
        # Adaptive exploration with density influence
        exploration = np.random.uniform(-self.exploration_factor, self.exploration_factor, 2)
        
        # Leading edge movement
        self.leading_velocity = (
            self.leading_velocity * self.persistence + 
            probing_force + 
            exploration
        )
        
        # Velocity management
        speed = np.linalg.norm(self.leading_velocity)
        max_speed = 0.015
        if speed > max_speed:
            self.leading_velocity *= max_speed / speed
        
        # Update leading edge
        self.leading_edge += self.leading_velocity
        
        # Nucleus-leading edge dynamics
        nucleus_to_lead_vec = self.leading_edge - self.nucleus
        self.nucleus_velocity = nucleus_to_lead_vec * 0.5
        self.nucleus += self.nucleus_velocity
        
        # Trailing edge update
        self.trailing_velocity = (self.nucleus - self.trailing_edge) * 0.4
        self.trailing_edge += self.trailing_velocity
        
        # Boundary conditions
        for point in [self.nucleus, self.leading_edge, self.trailing_edge]:
            point[:] = np.clip(point, 0, 1)

def create_packed_cell_distribution(
    total_cells=400, 
    packing_density=0.7
):
    """Create a densely packed cell distribution"""
    cells = []
    attempts = 0
    max_attempts = 10000
    
    while len(cells) < total_cells and attempts < max_attempts:
        # Generate candidate position
        candidate = np.random.uniform(0, 1, 2)
        
        # Check overlap with existing cells
        is_valid = True
        for cell in cells:
            dist = np.linalg.norm(candidate - cell.position)
            if dist < packing_density:  # Tight packing radius
                is_valid = False
                break
        
        if is_valid:
            cells.append(ElasticBubbleCell(candidate))
        
        attempts += 1
    
    return cells

def simulate_lymph_node(
    t_cells_count=100, 
    bubble_cells_count=600,
    packing_density=0.7
):
    filename = f'tcell_simulation-t{t_cells_count}-b{bubble_cells_count}-d{packing_density}.mp4'
    
    # Create densely packed bubble cells
    bubble_cells = create_packed_cell_distribution(bubble_cells_count, packing_density)
    
    # Place T cells within this dense environment
    t_cells = [
        TCell(bubble_cells[np.random.randint(len(bubble_cells))].position + 
              np.random.uniform(-0.02, 0.02, 2)) 
        for _ in range(t_cells_count)
    ]
    
    # Visualization setup
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_facecolor('black')
    ax.set_title('T Cell Migration through Dense Cellular Environment', color='white')

    # Scatter plot initialization
    bubble_scatter = ax.scatter([], [], c='lightblue', s=30, alpha=0.3)
    t_cell_nucleus = ax.scatter([], [], c='red', s=50, alpha=0.7)
    t_cell_leading = ax.scatter([], [], c='white', s=30, alpha=0.5)
    t_cell_trailing = ax.scatter([], [], c='blue', s=20, alpha=0.5)
    
    def update(frame):
        # Update bubble cell interactions to maintain density
        for cell in bubble_cells:
            cell.interact(bubble_cells)
        
        # Update T cell movements
        for t_cell in t_cells:
            t_cell.move(bubble_cells)
        
        # Position extraction
        bubble_pos = np.array([cell.position for cell in bubble_cells])
        t_cell_nucleus_pos = np.array([cell.nucleus for cell in t_cells])
        t_cell_leading_pos = np.array([cell.leading_edge for cell in t_cells])
        t_cell_trailing_pos = np.array([cell.trailing_edge for cell in t_cells])
        
        # Update scatter plots
        bubble_scatter.set_offsets(bubble_pos)
        t_cell_nucleus.set_offsets(t_cell_nucleus_pos)
        t_cell_leading.set_offsets(t_cell_leading_pos)
        t_cell_trailing.set_offsets(t_cell_trailing_pos)
        
        return [bubble_scatter, t_cell_nucleus, t_cell_leading, t_cell_trailing]

    # Animation creation
    ani = FuncAnimation(fig, update, frames=300, interval=50, blit=True)
    ani.save(filename, fps=20, dpi=150, writer='ffmpeg')
    plt.close(fig)


In [3]:
import os
import sys
import subprocess

# Diagnostic GPU Information Function
def print_gpu_info():
    """Print detailed GPU and CUDA information"""
    try:
        # CUDA version
        cuda_version = subprocess.check_output(["nvcc", "--version"]).decode('utf-8')
        print("CUDA Version:", cuda_version)
    except Exception as e:
        print("Could not retrieve CUDA version:", e)
    
    try:
        # GPU Information
        gpu_info = subprocess.check_output(["nvidia-smi"]).decode('utf-8')
        print("\nNVIDIA SMI Output:")
        print(gpu_info)
    except Exception as e:
        print("Could not retrieve GPU information:", e)

# Comprehensive TensorFlow GPU Setup
def setup_tensorflow_gpu():
    """
    Comprehensive TensorFlow GPU setup with detailed error handling
    """
    import tensorflow as tf
    
    try:
        # Clear any existing sessions
        tf.keras.backend.clear_session()
        
        # Print TensorFlow version and build configuration
        print("\nTensorFlow Version:", tf.__version__)
        print("Built with CUDA:", tf.test.is_built_with_cuda())
        
        # List all physical devices
        physical_devices = tf.config.list_physical_devices()
        print("\nAll Physical Devices:")
        for device in physical_devices:
            print(device)
        
        # Specifically check for GPUs
        gpus = tf.config.list_physical_devices('GPU')
        print("\nAvailable GPUs:")
        for gpu in gpus:
            print(gpu)
        
        # Ensure GPU memory growth
        if gpus:
            try:
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                print("\nMemory growth set successfully for all GPUs")
            except RuntimeError as e:
                print("\nError setting memory growth:", e)
                print("This might happen if GPU devices were initialized before this point.")
        
        # Verify GPU is being used
        print("\nCan see GPU:", tf.test.is_built_with_cuda())
        print("Can use GPU:", tf.test.is_gpu_available())
    
    except Exception as e:
        print("\nComprehensive GPU setup failed:")
        print(e)
        print("Detailed troubleshooting required.")

# Verify Prerequisites
def verify_gpu_prerequisites():
    """
    Check all prerequisites for GPU usage
    """
    # Check NVIDIA drivers
    try:
        driver_version = subprocess.check_output(["nvidia-smi", "--query-gpu=driver-version", "--format=csv,noheader"]).decode('utf-8').strip()
        print("\nNVIDIA Driver Version:", driver_version)
    except Exception as e:
        print("NVIDIA Driver check failed:", e)
        print("Suggested fix: Update NVIDIA drivers")
    
    # CUDA installation
    cuda_path = os.environ.get('CUDA_PATH', 'Not set')
    print("\nCUDA Path:", cuda_path)
    
    # Python and pip packages
    print("\nPython Packages:")
    packages = [
        "tensorflow", "tensorflow-gpu", "cuda-python", 
        "nvidia-cuda-runtime-cu11", "nvidia-cuda-nvcc-cu11"
    ]
    for pkg in packages:
        try:
            version = subprocess.check_output([sys.executable, "-m", "pip", "show", pkg]).decode('utf-8')
            print(f"{pkg} is installed")
        except subprocess.CalledProcessError:
            print(f"{pkg} is NOT installed")

def main():
    # Run diagnostic functions
    print_gpu_info()
    verify_gpu_prerequisites()
    
    # Setup TensorFlow GPU
    setup_tensorflow_gpu()
    
    # Rest of your machine learning code follows...
    import numpy as np
    import tensorflow as tf
    
    # Verify GPU usage in actual computation
    with tf.device('/GPU:0'):
        # Simple GPU computation test
        a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        b = tf.constant([[5.0, 6.0], [7.0, 8.0]])
        c = a * b
        print("\nGPU Computation Test:")
        print(c.numpy())

if __name__ == "__main__":
    main()


CUDA Version: nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Fri_Jan__6_16:45:21_PST_2023
Cuda compilation tools, release 12.0, V12.0.140
Build cuda_12.0.r12.0/compiler.32267302_0


NVIDIA SMI Output:
Tue Feb 17 10:50:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 2000 Ada Gene...    Off |   00000000:01:00.0  On |            

cuda-python is NOT installed
nvidia-cuda-runtime-cu11 is NOT installed


nvidia-cuda-nvcc-cu11 is NOT installed


2026-02-17 10:50:48.779638: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-17 10:50:49.126513: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-17 10:50:50.376079: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.



TensorFlow Version: 2.20.0
Built with CUDA: True

All Physical Devices:
PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

Available GPUs:
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

Memory growth set successfully for all GPUs

Can see GPU: True
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
Can use GPU: True

GPU Computation Test:
[[ 5. 12.]
 [21. 32.]]


I0000 00:00:1771285852.015854    9531 gpu_device.cc:2020] Created device /device:GPU:0 with 5497 MB memory:  -> device: 0, name: NVIDIA RTX 2000 Ada Generation Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1771285852.020904    9531 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5497 MB memory:  -> device: 0, name: NVIDIA RTX 2000 Ada Generation Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [15]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

class CellTrackingDataGenerator:
    def __init__(self, 
                 t_cells_count=50, 
                 bubble_cells_count=200, 
                 packing_density=0.05,
                 num_simulations=100,
                 grid_size=20,
                 time_steps=50):
        """
        Generate synthetic cell tracking datasets using ABM
        """
        self.t_cells_count = t_cells_count
        self.bubble_cells_count = bubble_cells_count
        self.packing_density = packing_density
        self.num_simulations = num_simulations
        self.grid_size = grid_size
        self.time_steps = time_steps
        
        # Tracking data storage
        self.tracking_data = []
        self.labels = []
    
    def generate_abm_tracking_data(self):
        """
        Generate synthetic tracking data using the ABM simulation
        """
        # Import these here to avoid potential circular import
        # defined in above cell, no need to import for now
        # from your_abm_module import create_packed_cell_distribution, TCell
        
        all_simulation_frames = []
        
        for _ in range(self.num_simulations):
            # Create simulation environment
            bubble_cells = create_packed_cell_distribution(
                total_cells=self.bubble_cells_count, 
                packing_density=self.packing_density
            )
            
            # Place T cells
            t_cells = [
                TCell(bubble_cells[np.random.randint(len(bubble_cells))].position + 
                      np.random.uniform(-0.02, 0.02, 2)) 
                for _ in range(self.t_cells_count)
            ]
            
            # Collect frames of cell positions
            simulation_frames = []
            for _ in range(50):  # 50 time steps
                # Update bubble cells and T cells
                for cell in bubble_cells:
                    cell.interact(bubble_cells)
                
                for t_cell in t_cells:
                    t_cell.move(bubble_cells)
                
                # Extract positions
                frame = {
                    'bubble_cells': np.array([cell.position for cell in bubble_cells]),
                    't_cell_nuclei': np.array([cell.nucleus for cell in t_cells]),
                    't_cell_leading': np.array([cell.leading_edge for cell in t_cells]),
                    't_cell_trailing': np.array([cell.trailing_edge for cell in t_cells])
                }
                simulation_frames.append(frame)
            
            all_simulation_frames.append(simulation_frames)
        
        return all_simulation_frames

    def prepare_ml_dataset(self):
        simulation_data = self.generate_abm_tracking_data()
        
        X = []  # Input features
        y = []  # Target trajectories
        
        # Collect ALL positions to get truly global normalization
        all_positions = []
        for simulation in simulation_data:
            for frame in simulation:
                all_positions.extend(frame['t_cell_nuclei'])
        
        # Global normalization parameters
        global_min = np.min(all_positions, axis=0)
        global_max = np.max(all_positions, axis=0)
        
        for simulation in simulation_data:
            sim_frames = []
            sim_trajectories = []
            
            for frame in simulation:
                # IMPROVED Normalization
                def robust_normalize(positions):
                    if len(positions) == 0:
                        return np.zeros((self.t_cells_count, 2))
                    
                    # Ensure consistent shape WITHOUT zero-padding
                    positions = positions[:self.t_cells_count]
                    
                    # Robust normalization across ALL simulations
                    normalized = (positions - global_min) / (global_max - global_min)
                    
                    # Handle potential edge cases
                    normalized = np.clip(normalized, 0, 1)
                    
                    return normalized
                
                # Create grid representation
                grid = np.zeros((self.grid_size, self.grid_size, 4), dtype=np.float32)
                
                # Use robust normalization
                grid[:,:,0] = self._create_density_map(robust_normalize(frame['bubble_cells']))
                grid[:,:,1] = self._create_density_map(robust_normalize(frame['t_cell_nuclei']))
                grid[:,:,2] = self._create_density_map(robust_normalize(frame['t_cell_leading']))
                grid[:,:,3] = self._create_density_map(robust_normalize(frame['t_cell_trailing']))
                
                sim_frames.append(grid)
                
                # Store normalized trajectories
                sim_trajectories.append(robust_normalize(frame['t_cell_nuclei']))
            
            X.append(sim_frames)
            y.append(np.array(sim_trajectories))
        
        # Convert to numpy arrays
        X = np.array(X)
        y = np.array(y)
        
        # Detailed debugging
        print("Simulation Data Overview:")
        print("Total Simulations:", len(simulation_data))
        print("X shape:", X.shape)
        print("y shape:", y.shape)
        print("\nGlobal Normalization:")
        print("Global Min:", global_min)
        print("Global Max:", global_max)

        # plot out
        self.visualize_raw_tracks(simulation_data)
        
        # Optional: More rigorous sanity checks
        self._validate_dataset(X, y)
        
        return X, y
    
    def _create_density_map(self, normalized_positions):
        """
        Create a smoother density representation
        """
        grid_map = np.zeros((self.grid_size, self.grid_size))
        
        # Use Gaussian-like distribution instead of discrete mapping
        for pos in normalized_positions:
            x = pos[0] * (self.grid_size - 1)
            y = pos[1] * (self.grid_size - 1)
            
            # Spread influence around the point
            x_int, y_int = int(x), int(y)
            
            # Simple weighted distribution
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    weight = 1 - np.sqrt(dx**2 + dy**2) / np.sqrt(2)
                    if 0 <= x_int+dx < self.grid_size and 0 <= y_int+dy < self.grid_size:
                        grid_map[x_int+dx, y_int+dy] += weight
        
        # Normalize
        return grid_map / (np.max(grid_map) if np.max(grid_map) > 0 else 1)
    
    def _validate_dataset(self, X, y):
        """
        Additional sanity checks for the dataset
        """
        # Check for NaN or infinite values
        assert not np.isnan(X).any(), "Dataset contains NaN values"
        assert not np.isnan(y).any(), "Trajectory contains NaN values"
        
        # Check value ranges
        assert np.all((X >= 0) & (X <= 1)), "X values out of [0,1] range"
        assert np.all((y >= 0) & (y <= 1)), "Trajectory values out of [0,1] range"
        
        # Check trajectory consistency
        trajectory_variations = np.std(y, axis=1)
        print("\nTrajectory Variation Statistics:")
        print("Mean Variation:", np.mean(trajectory_variations))
        print("Max Variation:", np.max(trajectory_variations))
    
    def visualize_raw_tracks(self, simulation_data):
        """
        Visualize raw tracks from the ABM simulation
        """
        plt.figure(figsize=(15, 10))
        
        # Plot trajectories for multiple simulations
        for sim_idx in range(min(5, len(simulation_data))):
            plt.subplot(2, 3, sim_idx + 1)
            
            # Extract T cell nucleus positions for this simulation
            simulation = simulation_data[sim_idx]
            t_cell_tracks = []
            
            # Collect T cell nucleus positions for each frame
            for frame in simulation:
                t_cell_positions = frame['t_cell_nuclei']
                t_cell_tracks.append(t_cell_positions)
            
            # Convert to numpy array for easier manipulation
            t_cell_tracks = np.array(t_cell_tracks)
            
            # Plot each T cell's trajectory
            for cell_idx in range(t_cell_tracks.shape[1]):
                plt.plot(
                    t_cell_tracks[:, cell_idx, 0], 
                    t_cell_tracks[:, cell_idx, 1], 
                    label=f'Cell {cell_idx}'
                )
            
            plt.title(f'Simulation {sim_idx} T Cell Tracks')
            plt.xlabel('X Position')
            plt.ylabel('Y Position')
        
        plt.tight_layout()
        plt.savefig('model_outputs/raw_cell_tracks.png')
        plt.close()

class ImprovedCellTrackingModel:
    def __init__(self, input_shape=(50, 20, 20, 4), t_cells_count=50):
        self.input_shape = input_shape
        self.t_cells_count = t_cells_count
        
        # Create attention layers outside the model building
        self.attention_dense = tf.keras.layers.Dense(1, activation='tanh')
        
        self.model = self.build_advanced_model()

    def custom_attention_layer(self, inputs):
        """
        Attention mechanism implemented as a method
        """
        # Compute attention scores
        attention_scores = self.attention_dense(inputs)
        
        # Softmax along time steps
        attention_weights = tf.nn.softmax(attention_scores, axis=1)
        
        # Apply attention
        context = inputs * attention_weights
        
        # Sum across time steps
        return tf.reduce_sum(context, axis=1)
    
    def residual_block(self, x, filters, kernel_size=(3, 3)):
        """
        Create a residual block for improved feature extraction
        """
        # Store the original input
        residual = x
        
        # If the input and output have different numbers of filters, 
        # use a 1x1 convolution to match dimensions
        if residual.shape[-1] != filters:
            residual = tf.keras.layers.TimeDistributed(
                tf.keras.layers.Conv2D(filters, (1, 1), padding='same')
            )(residual)
        
        # Convolutional layers
        x = tf.keras.layers.TimeDistributed(
            tf.keras.layers.Conv2D(filters, kernel_size, 
                                   activation='relu', 
                                   padding='same')
        )(x)
        x = tf.keras.layers.TimeDistributed(
            tf.keras.layers.BatchNormalization()
        )(x)
        x = tf.keras.layers.TimeDistributed(
            tf.keras.layers.Conv2D(filters, kernel_size, 
                                   activation=None, 
                                   padding='same')
        )(x)
        
        # Add residual connection
        x = tf.keras.layers.Add()([residual, x])
        x = tf.keras.layers.Activation('relu')(x)
        return x
    
    def build_advanced_model(self):
        def advanced_trajectory_loss(y_true, y_pred):
            """
            Robust trajectory loss function
            """
            # Ensure consistent shapes
            y_true = tf.cast(y_true, tf.float32)
            y_pred = tf.cast(y_pred, tf.float32)
            
            # Handling potential shape inconsistencies
            y_true = y_true[:, :y_pred.shape[1], :, :]
            
            # Euclidean distance loss
            distance_loss = tf.reduce_mean(
                tf.sqrt(
                    tf.reduce_sum(
                        tf.square(y_true - y_pred), 
                        axis=-1
                    )
                )
            )
            
            # Trajectory smoothness loss
            # Ensure we have at least 2 time steps
            if y_pred.shape[1] > 1:
                trajectory_diff = y_pred[:, 1:] - y_pred[:, :-1]
                smoothness_loss = tf.reduce_mean(tf.square(trajectory_diff))
            else:
                smoothness_loss = tf.constant(0.0)
            
            # Velocity consistency loss
            if y_pred.shape[1] > 1:
                velocity_true = y_true[:, 1:] - y_true[:, :-1]
                velocity_pred = y_pred[:, 1:] - y_pred[:, :-1]
                velocity_loss = tf.reduce_mean(tf.square(velocity_true - velocity_pred))
            else:
                velocity_loss = tf.constant(0.0)
            
            # Combined loss with adjustable weights
            return (distance_loss + 
                    0.1 * smoothness_loss + 
                    0.05 * velocity_loss)

        inputs = tf.keras.layers.Input(shape=self.input_shape)
        
        # Initial feature extraction with residual blocks
        x = inputs
        x = self.residual_block(x, 64)
        x = self.residual_block(x, 64)
        
        # Spatial-temporal feature extraction
        x = tf.keras.layers.TimeDistributed(
            tf.keras.layers.MaxPooling2D((2, 2))
        )(x)
        
        x = tf.keras.layers.TimeDistributed(
            tf.keras.layers.Flatten()
        )(x)
        
        # Bidirectional LSTM with attention
        x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(128, 
                                 return_sequences=True, 
                                 dropout=0.3, 
                                 recurrent_dropout=0.3)
        )(x)
        
        # Improved attention mechanism
        x = tf.keras.layers.Lambda(self.custom_attention_layer)(x)
        
        # Trajectory prediction layers
        x = tf.keras.layers.Dense(256, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.4)(x)
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        
        # Final trajectory output
        outputs = tf.keras.layers.Dense(
            self.t_cells_count * 2, 
            activation='linear'
        )(x)
        
        outputs = tf.keras.layers.Reshape(
            (-1, self.t_cells_count, 2)
        )(outputs)
        
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=1e-3, 
                weight_decay=1e-5
            ),
            loss=advanced_trajectory_loss,
            metrics=['mae']
        )
        
        return model
    
    def train(self, X, y, verbose=1):
        # Learning rate scheduling
        lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.5, 
            patience=7, 
            min_lr=1e-5
        )
        
        # Early stopping
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=15, 
            restore_best_weights=True
        )
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        # Train with advanced callbacks
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=100,
            batch_size=16,
            callbacks=[lr_schedule, early_stop],
            verbose=verbose
        )
        
        return history
    
    def visualize_tracks(self, X_test, y_test):
        """
        Advanced track visualization
        """
        # Predict tracks
        predicted_tracks = self.model.predict(X_test)
        
        plt.figure(figsize=(20, 10))
        
        # Original tracks
        plt.subplot(1, 2, 1)
        for cell_idx in range(y_test.shape[1]):
            plt.plot(y_test[0, cell_idx, :, 0], y_test[0, cell_idx, :, 1], 
                     label=f'Cell {cell_idx}')
        plt.title('Original Tracks')
        plt.xlabel('Normalized X')
        plt.ylabel('Normalized Y')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Predicted tracks
        plt.subplot(1, 2, 2)
        for cell_idx in range(predicted_tracks.shape[2]):
            plt.plot(predicted_tracks[0, :, cell_idx, 0], 
                     predicted_tracks[0, :, cell_idx, 1], 
                     label=f'Cell {cell_idx}')
        plt.title('Predicted Tracks')
        plt.xlabel('Normalized X')
        plt.ylabel('Normalized Y')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        plt.tight_layout()
        plt.savefig('model_outputs/cell_tracking_comparison.png', bbox_inches='tight')
        plt.close()

        # Additional detailed visualization
        plt.figure(figsize=(20, 10))
        
        # Side-by-side comparison for each cell
        for cell_idx in range(min(5, y_test.shape[1])):
            plt.subplot(2, 5, cell_idx + 1)
            plt.plot(y_test[0, cell_idx, :, 0], y_test[0, cell_idx, :, 1], 
                     label='Original', color='blue')
            plt.title(f'Cell {cell_idx} Original')
            
            plt.subplot(2, 5, cell_idx + 6)
            plt.plot(predicted_tracks[0, :, cell_idx, 0], 
                     predicted_tracks[0, :, cell_idx, 1], 
                     label='Predicted', color='red')
            plt.title(f'Cell {cell_idx} Predicted')
        
        plt.tight_layout()
        plt.savefig('model_outputs/individual_cell_tracks.png', bbox_inches='tight')
        plt.close()
    
    def performance_analysis(self, X_test, y_test):
        """
        Detailed performance metrics
        """
        predicted_tracks = self.model.predict(X_test)
        
        # Mean Squared Error per cell
        mse_per_cell = np.mean(np.square(y_test - predicted_tracks), axis=(0,1,3))
        print("\nMean Squared Error per Cell:")
        for idx, mse in enumerate(mse_per_cell):
            print(f"Cell {idx}: {mse}")
        
        # Total tracking error
        total_tracking_error = np.mean(np.abs(y_test - predicted_tracks))
        print(f"\nTotal Tracking Error: {total_tracking_error}")

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Ensure output directory
os.makedirs('model_outputs', exist_ok=True)

# Data Generation (use your existing CellTrackingDataGenerator)
data_generator = CellTrackingDataGenerator(
    t_cells_count=50,
    bubble_cells_count=200,
    packing_density=0.05,
    num_simulations=12  # Increased simulations
)

# Prepare dataset
X, y = data_generator.prepare_ml_dataset()

# Create improved tracking model
tracking_model = ImprovedCellTrackingModel()

# Train model
history = tracking_model.train(X, y)

# Visualize tracks
tracking_model.visualize_tracks(X[:10], y[:10])

# Performance analysis
tracking_model.performance_analysis(X[:10], y[:10])


Simulation Data Overview:
Total Simulations: 12
X shape: (12, 50, 20, 20, 4)
y shape: (12, 50, 50, 2)

Global Normalization:
Global Min: [0. 0.]
Global Max: [1. 1.]

Trajectory Variation Statistics:
Mean Variation: 0.04061662029540607
Max Variation: 0.11227070921351022
Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 23s 23s/step - loss: 0.8321 - mae: 0.5441 - val_loss: 0.7407 - val_mae: 0.4770 - learning_rate: 0.0010
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 462ms/step - loss: 0.7870 - mae: 0.5086 - val_loss: 0.6970 - val_mae: 0.4506 - learning_rate: 0.0010
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step - loss: 0.7206 - mae: 0.4676 - val_loss: 0.6653 - val_mae: 0.4288 - learning_rate: 0.0010
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step - loss: 0.6905 - mae: 0.4490 - val_loss: 0.6292 - val_mae: 0.4043 - learning_rate: 0.0010
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 442ms/step - loss: 0.6516 - mae: 0.4189 - val_loss: 0.5879 - val_mae: 0.3756 - learning_rate: 0.0010
Epoch 6/100
1/1 ━━━━━━━━━━

In [ ]:
tracking_model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

class CellTrackPredictionEvaluator:
    def __init__(self, trained_model, grid_size=20):
        """
        Initialize prediction evaluator
        
        Parameters:
        - trained_model: Trained Keras model for T cell tracking
        - grid_size: Size of the grid used in training
        """
        self.model = trained_model
        self.grid_size = grid_size
    
    def prepare_abm_data_for_prediction(self, t_cells, bubble_cells, num_frames=50):
        """
        Convert ABM simulation data to model input format
        
        Key Change: Capture initial positions explicitly
        """
        # Store initial positions before any movement
        initial_positions = np.array([cell.nucleus for cell in t_cells])
        
        # Create input tensor
        X = np.zeros((1, num_frames, self.grid_size, self.grid_size, 4), dtype=np.float32)
        
        # Helper function to map positions to grid
        def map_to_grid(positions, channel):
            # Scale positions to grid coordinates
            scaled_pos = (np.array(positions) * self.grid_size).astype(int)
            scaled_pos = np.clip(scaled_pos, 0, self.grid_size-1)
            
            # Mark positions on specific channel
            for x, y in scaled_pos:
                X[0, frame, x, y, channel] = 1.0
        
        # Simulate and record frames
        for frame in range(num_frames):
            # Update bubble cell interactions
            for cell in bubble_cells:
                cell.interact(bubble_cells)
            
            # Update T cell movements
            for t_cell in t_cells:
                t_cell.move(bubble_cells)
            
            # Map cell positions to grid
            map_to_grid([cell.position for cell in bubble_cells], 0)      # Bubble cells
            map_to_grid([cell.nucleus for cell in t_cells], 1)            # T cell nuclei
            map_to_grid([cell.leading_edge for cell in t_cells], 2)       # Leading edges
            map_to_grid([cell.trailing_edge for cell in t_cells], 3)      # Trailing edges
        
        return X, initial_positions
    
    def extract_true_tracks(self, t_cells, num_frames=50):
        """
        Extract true T cell tracks from ABM simulation
        
        Parameters:
        - t_cells: List of T cells from ABM simulation
        - num_frames: Number of frames to process
        
        Returns:
        - True tracks for each T cell
        """
        # Track nuclei positions over time
        true_tracks = []
        for t_cell in t_cells:
            cell_track = []
            for _ in range(num_frames):
                cell_track.append(t_cell.nucleus.copy())
                t_cell.move([])  # Move without bubble cells for track extraction
            true_tracks.append(cell_track)
        
        return np.array(true_tracks)
    
    def evaluate_tracking_performance(self, true_tracks, predicted_tracks):
        """
        Compute detailed tracking performance metrics
        
        Parameters:
        - true_tracks: Original T cell tracks from ABM
        - predicted_tracks: Model-predicted tracks
        
        Returns:
        - Dictionary of performance metrics
        """
        # Reshape tracks for comparison
        true_tracks_reshaped = true_tracks.reshape(true_tracks.shape[0], -1, 2)
        predicted_tracks_reshaped = predicted_tracks.reshape(predicted_tracks.shape[0], -1, 2)
        
        # Compute metrics
        metrics = {
            'Mean Absolute Error (MAE)': np.mean(np.abs(true_tracks_reshaped - predicted_tracks_reshaped)),
            'Root Mean Squared Error (RMSE)': np.sqrt(np.mean(np.square(true_tracks_reshaped - predicted_tracks_reshaped))),
            'Mean Tracking Error': np.mean(np.linalg.norm(true_tracks_reshaped - predicted_tracks_reshaped, axis=2)),
            'Tracking Consistency': np.mean(np.all(np.abs(true_tracks_reshaped - predicted_tracks_reshaped) < 0.05, axis=(1,2)))
        }
        
        return metrics
    
    def visualize_tracks(self, true_tracks, predicted_tracks, num_cells=5):
        """
        Create comparative visualization of true and predicted tracks
        
        Parameters:
        - true_tracks: Original T cell tracks from ABM
        - predicted_tracks: Model-predicted tracks
        - num_cells: Number of cells to visualize
        """
        plt.figure(figsize=(15, 6))
        
        # True tracks subplot
        plt.subplot(1, 2, 1)
        true_tracks_reshaped = true_tracks.reshape(true_tracks.shape[0], -1, 2)
        for i in range(min(num_cells, true_tracks_reshaped.shape[0])):
            plt.plot(true_tracks_reshaped[i, :, 0], true_tracks_reshaped[i, :, 1], 
                     label=f'True Track {i+1}', marker='o', markersize=3)
        plt.title('True T Cell Tracks')
        plt.xlabel('X Position')
        plt.ylabel('Y Position')
        plt.legend()
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        
        # Predicted tracks subplot
        plt.subplot(1, 2, 2)
        predicted_tracks_reshaped = predicted_tracks.reshape(predicted_tracks.shape[0], -1, 2)
        for i in range(min(num_cells, predicted_tracks_reshaped.shape[0])):
            plt.plot(predicted_tracks_reshaped[i, :, 0], predicted_tracks_reshaped[i, :, 1], 
                     label=f'Predicted Track {i+1}', marker='o', markersize=3)
        plt.title('Predicted T Cell Tracks')
        plt.xlabel('X Position')
        plt.ylabel('Y Position')
        plt.legend()
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        
        plt.tight_layout()
        plt.savefig('model_outputs/t_cell_track_comparison.png')
        plt.close()

def main():
    # Set random seeds for reproducibility
    np.random.seed(42)
    tf.random.set_seed(42)
    
    # Create ABM simulation
    # Adjust parameters as needed
    t_cells_count = 50
    bubble_cells_count = 200
    packing_density = 0.05
    
    # Create bubble cells and T cells
    bubble_cells = create_packed_cell_distribution(
        total_cells=bubble_cells_count, 
        packing_density=packing_density
    )
    
    # Place T cells
    t_cells = [
        TCell(bubble_cells[np.random.randint(len(bubble_cells))].position + 
              np.random.uniform(-0.02, 0.02, 2)) 
        for _ in range(t_cells_count)
    ]
    
    # Initialize prediction evaluator
    track_evaluator = CellTrackPredictionEvaluator(tracking_model.model)
    
    # Prepare input data for prediction
    X_test, initial_positions = track_evaluator.prepare_abm_data_for_prediction(t_cells, bubble_cells)
    
    # Extract true tracks
    true_tracks = track_evaluator.extract_true_tracks(t_cells)
    
    # Predict tracks
    predicted_tracks = track_evaluator.model.predict(X_test)
    
    # Reshape predicted tracks
    predicted_tracks = predicted_tracks.reshape(true_tracks.shape)
    
    # Align starting positions
    for i in range(predicted_tracks.shape[0]):
        predicted_tracks[i, 0, :] = initial_positions[i]
    
    # Visualize tracks
    track_evaluator.visualize_tracks(true_tracks, predicted_tracks)
    
    # Evaluate performance
    performance_metrics = track_evaluator.evaluate_tracking_performance(true_tracks, predicted_tracks)
    
    # Print performance metrics
    print("\nTracking Performance Metrics:")
    for metric, value in performance_metrics.items():
        print(f"{metric}: {value}")

# Run the main function
if __name__ == '__main__':
    main()
